# ARCADE Stenosis Segmentation v2 — Closing the Gap to StenUNet

## Where things stand

The first segmentation run (once the dataset-path bug was fixed) gave real, working numbers:

| Method | Pixel Dice | Precision | Recall | F1 |
|---|---|---|---|---|
| U-Net | 0.5287 | 0.2420 | 0.8763 | 0.3793 |
| Attention U-Net | 0.5362 | 0.2585 | 0.8421 | 0.3956 |
| StenUNet (target) | — | — | — | 0.5350 |

Both models already beat the YOLO baseline (F1 0.329) and match/slightly beat the official ARCADE YOLOv8 baseline
(F1 0.38). The clear pattern in the numbers: **recall is already very high (0.84-0.88), precision is low
(0.24-0.26)** — the model finds most real lesions but also flags a lot of non-lesion tissue. Closing the gap to
StenUNet's 0.535 means raising precision substantially without giving back much of that recall, and improving pixel
Dice/IoU generally. This notebook makes four targeted, individually-tested changes aimed directly at that, rather
than re-running the same recipe for longer or guessing at knobs.

**Every change below was unit-tested against synthetic tensors before being put in this notebook** — shapes,
loss values, and the flip-TTA averaging logic were all verified to behave correctly in isolation first.

## What's changing and why (each one targets something specific in the numbers above)

1. **Deep supervision** (the architecture itself). StenUNet's nnU-Net framework uses auxiliary loss heads at
   multiple decoder resolutions, not just the final output — this pushes useful gradient into the earlier decoder
   stages directly instead of relying on it propagating all the way back from the last layer, which generally
   improves both convergence and final Dice on segmentation tasks. Both `UNet` and `AttentionUNet` now have three
   auxiliary 1×1-conv heads at H/2, H/4, and H/8 resolution in addition to the main full-resolution output, each
   supervised against a resolution-matched downsampled version of the ground-truth mask, combined with nnU-Net's own
   weighting scheme (finer resolutions weighted more: 1, 0.5, 0.25, 0.125).
2. **Tversky loss replacing plain Dice, with `alpha=0.6 > beta=0.4`** (deliberately, not the usual medical-imaging
   default of `beta > alpha`). Tversky loss is Dice generalized with separate, tunable weights on false positives
   (`alpha`) and false negatives (`beta`); weighting FP higher than FN directly targets the observed problem — low
   precision, i.e. too many false positives — rather than pushing recall even higher, which is the wrong direction
   given where the numbers already are.
3. **`POS_WEIGHT` lowered from 10 to 5** in the BCE loss component, for the same reason: 10 was the validated choice
   from a different project (stenoz) whose own precision/recall balance we don't know — it isn't a fixed law, and
   the actual observed imbalance here (recall far above precision) says it's currently pulling too hard toward
   recall.
4. **Two inference-time additions that need no retraining and carry very low risk**: horizontal-flip test-time
   augmentation (predict on the image and its mirror, average the two probability maps — cancels some
   orientation-specific false positives), and a small-connected-component filter that removes predicted blobs
   smaller than a threshold set from the *training data's own* smallest real lesions (not an arbitrary number) —
   directly aimed at precision, since spurious tiny predicted regions are exactly what drag precision down without
   costing real recall.
5. **A U-Net + Attention U-Net ensemble** (simple probability averaging of both trained models) is added as a third
   evaluated variant in the final table, since ensembling two models that individually differ in their
   precision/recall trade-off often lands on a better F1 than either alone — reported honestly alongside the
   individual models, not presented as if it were free.

**Still no guarantee of beating 0.535** — StenUNet's full recipe includes things this notebook still doesn't
(auto-configured architecture search, multi-preprocessing-variant ensembling at inference). What's here is a
coherent, evidence-driven step up from the first run, each piece justified by what the first run's own numbers
showed, and it will be reported exactly as it comes out either way.


## 1. Install & Import Dependencies

In [ ]:
# ============================================================
# 1a. Installs
# ============================================================
!pip install -q pycocotools scikit-image kagglehub albumentations
print("Installs complete.")


In [ ]:
# ============================================================
# 1b. Imports
# ============================================================
import os
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A

from skimage.measure import label as cc_label, regionprops

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Get the ARCADE Dataset onto the Colab Runtime

In [ ]:
# ============================================================
# 2a. Download dataset via kagglehub
# ============================================================
import kagglehub

download_path = Path(kagglehub.dataset_download("nirmalgaud/arcade-dataset"))
candidates = list(download_path.rglob("train.json"))
if not candidates:
    raise FileNotFoundError(f"Could not find train.json under {download_path}. Browse manually and set DATASET_ROOT.")

# ARCADE packages multiple sub-tasks (stenosis localization, syntax/vessel segmentation, etc.) --
# filter explicitly for "stenosis" rather than trusting glob order (this bit a previous run: it
# silently resolved to .../arcade/syntax, whose category ids don't include stenosis at all).
stenosis_candidates = [c for c in candidates if "stenosis" in str(c).lower()]
if not stenosis_candidates:
    raise FileNotFoundError(
        f"Found train.json files under {download_path}, but none under a 'stenosis' folder: "
        f"{[str(c) for c in candidates]}. Inspect the paths and set DATASET_ROOT manually."
    )
DATASET_ROOT = stenosis_candidates[0].parent.parent.parent
print("Resolved DATASET_ROOT:", DATASET_ROOT)
assert "stenosis" in str(DATASET_ROOT).lower()

for split in ["train", "val", "test"]:
    assert (DATASET_ROOT / split / "annotations" / f"{split}.json").exists()
    assert (DATASET_ROOT / split / "images").exists()
print("Dataset structure verified.")


In [ ]:
# ============================================================
# 2b. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/arcade_stenosis_runs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive output dir:", DRIVE_OUTPUT_DIR)


## 3. Configuration

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
WORK_ROOT = Path("/content/stenosis_seg_v2")
MASK_ROOT = WORK_ROOT / "masks"

SPLITS = ["train", "val", "test"]
STENOSIS_CATEGORY_ID = 26

IMG_SIZE = 512
BASE_CHANNELS = 32
BATCH_SIZE = 8
EPOCHS = 150             # raised from 100 -- v1's val_dice curve was still climbing at epoch 100
PATIENCE = 30             # raised from 20, to match
LR = 1e-3

POS_WEIGHT = 5.0          # lowered from v1's 10 -- see Section 0 rationale
TVERSKY_ALPHA = 0.6       # weight on False Positives
TVERSKY_BETA = 0.4        # weight on False Negatives -- alpha > beta targets v1's low-precision problem
DS_SCALE_WEIGHTS = (1.0, 0.5, 0.25, 0.125)   # deep supervision weights: main, H/2, H/4, H/8

USE_FLIP_TTA = True
MODELS_TO_TRAIN = ["unet", "attention_unet"]

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: no GPU detected -- this notebook trains two deep-supervision models; on CPU this will be very slow.")

for p in [WORK_ROOT, MASK_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
print("Config set. Device:", DEVICE)


## 4. Build Pixel-Level Masks from Polygon Annotations

Unchanged from the working v1 pipeline, including the fail-fast sanity check that catches a wrong `DATASET_ROOT`
before any GPU time is spent.


In [ ]:
# ============================================================
# 4. Rasterize polygons into masks
# ============================================================
def load_coco(split):
    with open(DATASET_ROOT / split / "annotations" / f"{split}.json") as f:
        return json.load(f)

coco_data = {split: load_coco(split) for split in SPLITS}

def polygon_to_mask(segmentation, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
        pts = np.round(pts).astype(np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(mask, [pts], 1)
    return mask

def build_masks(split):
    coco = coco_data[split]
    img_dir = DATASET_ROOT / split / "images"
    out_img_dir = MASK_ROOT / split / "images"
    out_mask_dir = MASK_ROOT / split / "masks"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        if ann["category_id"] == STENOSIS_CATEGORY_ID:
            anns_by_image[ann["image_id"]].append(ann)

    for im in tqdm(coco["images"], desc=f"Building masks ({split})"):
        img = cv2.imread(str(img_dir / im["file_name"]), cv2.IMREAD_COLOR)
        h, w = im["height"], im["width"]
        mask = np.zeros((h, w), dtype=np.uint8)
        for ann in anns_by_image.get(im["id"], []):
            seg = ann.get("segmentation", [])
            if seg:
                mask |= polygon_to_mask(seg, h, w)

        img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        mask_r = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        stem = Path(im["file_name"]).stem
        cv2.imwrite(str(out_img_dir / f"{stem}.png"), img_r)
        cv2.imwrite(str(out_mask_dir / f"{stem}.png"), mask_r * 255)

for split in SPLITS:
    build_masks(split)
print("Masks built under:", MASK_ROOT)

# Fail fast if masks came out empty -- this exact silent failure has happened once already.
_train_mask_files = os.listdir(MASK_ROOT / "train" / "masks")
_n_nonempty = sum(1 for f in _train_mask_files if cv2.imread(str(MASK_ROOT / "train" / "masks" / f), cv2.IMREAD_GRAYSCALE).max() > 0)
_pct_nonempty = 100 * _n_nonempty / len(_train_mask_files)
print(f"Sanity check: {_n_nonempty}/{len(_train_mask_files)} train masks ({_pct_nonempty:.1f}%) contain at least one lesion pixel.")
assert _pct_nonempty > 80, (
    f"Only {_pct_nonempty:.1f}% of train masks contain any lesion pixels -- DATASET_ROOT is almost certainly "
    "pointing at the wrong ARCADE sub-task folder. Check Section 2's 'Resolved DATASET_ROOT' print."
)


In [ ]:
# ============================================================
# 4b. Data-driven minimum-component-area threshold for the post-processing filter (Section 9)
#     -- computed from the smallest real lesions in the TRAINING masks, not guessed.
# ============================================================
train_lesion_areas = []
for f in os.listdir(MASK_ROOT / "train" / "masks"):
    m = cv2.imread(str(MASK_ROOT / "train" / "masks" / f), cv2.IMREAD_GRAYSCALE)
    cc = cc_label((m > 127).astype(np.uint8))
    for r in regionprops(cc):
        train_lesion_areas.append(r.area)

train_lesion_areas = np.array(train_lesion_areas)
MIN_COMPONENT_AREA = int(np.percentile(train_lesion_areas, 5))   # 5th percentile of real lesion sizes
print(f"Real lesion pixel-area stats (train): min={train_lesion_areas.min()}, "
      f"5th pct={np.percentile(train_lesion_areas, 5):.0f}, median={np.median(train_lesion_areas):.0f}")
print(f"MIN_COMPONENT_AREA set to {MIN_COMPONENT_AREA}px -- predicted blobs smaller than this "
      "are filtered out as noise in Section 9, since they're smaller than 95% of real lesions in the training data.")


## 5. Dataset & Augmentation

In [ ]:
# ============================================================
# 5a. Augmentation -- unchanged from v1 (already reasoned through and working)
# ============================================================
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(rotate=(-10, 10), translate_percent=(0.0, 0.10), scale=(0.85, 1.15), shear=(-2, 2), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(p=0.2),
])
val_transform = None

class StenosisSegDataset(Dataset):
    def __init__(self, split, transform=None):
        self.img_dir = MASK_ROOT / split / "images"
        self.mask_dir = MASK_ROOT / split / "masks"
        self.files = sorted(os.listdir(self.img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = cv2.cvtColor(cv2.imread(str(self.img_dir / fname)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_dir / fname), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform is not None:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]
        img = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return img, mask, fname

train_ds = StenosisSegDataset("train", transform=train_transform)
val_ds = StenosisSegDataset("val", transform=val_transform)
test_ds = StenosisSegDataset("test", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


## 6. Model Definitions — U-Net & Attention U-Net, Both With Deep Supervision

Both networks now return a list of four outputs: `[main (full-res), aux_H/2, aux_H/4, aux_H/8]`. Shapes for this were
verified against synthetic tensors before being used here.


In [ ]:
# ============================================================
# 6a. Building blocks (unchanged) + deep-supervision heads (new)
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.pool_conv(x)


class AttentionGate(nn.Module):
    """Oktay et al. 2018."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        return x * self.psi(psi)


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, use_attention=False):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.use_attention = use_attention
        if use_attention:
            self.attn = AttentionGate(F_g=in_ch // 2, F_l=skip_ch, F_int=skip_ch // 2)
        self.conv = DoubleConv(in_ch // 2 + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        if self.use_attention:
            skip = self.attn(g=x, x=skip)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class DeepSupNet(nn.Module):
    """U-Net (use_attention=False) or Attention U-Net (use_attention=True), both with nnU-Net-style
       deep supervision: three auxiliary 1x1-conv heads at H/2, H/4, H/8, plus the full-res main output.
       forward() returns [main, aux_H/2, aux_H/4, aux_H/8] -- always take output[0] for inference."""
    def __init__(self, in_ch=3, out_ch=1, base=BASE_CHANNELS, use_attention=False):
        super().__init__()
        c1, c2, c3, c4, c5 = base, base*2, base*4, base*8, base*16
        self.inc = DoubleConv(in_ch, c1)
        self.down1 = Down(c1, c2); self.down2 = Down(c2, c3); self.down3 = Down(c3, c4); self.down4 = Down(c4, c5)
        self.up1 = Up(c5, c4, c4, use_attention); self.up2 = Up(c4, c3, c3, use_attention)
        self.up3 = Up(c3, c2, c2, use_attention); self.up4 = Up(c2, c1, c1, use_attention)
        self.outc = nn.Conv2d(c1, out_ch, 1)
        self.ds_deep = nn.Conv2d(c4, out_ch, 1)     # at u1 (H/8)
        self.ds_mid = nn.Conv2d(c3, out_ch, 1)      # at u2 (H/4)
        self.ds_shallow = nn.Conv2d(c2, out_ch, 1)  # at u3 (H/2)

    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        u1 = self.up1(x5, x4)
        u2 = self.up2(u1, x3)
        u3 = self.up3(u2, x2)
        u4 = self.up4(u3, x1)
        return [self.outc(u4), self.ds_shallow(u3), self.ds_mid(u2), self.ds_deep(u1)]


MODEL_CLASSES = {
    "unet": lambda: DeepSupNet(use_attention=False),
    "attention_unet": lambda: DeepSupNet(use_attention=True),
}

# shape sanity check before committing to a full training run
for name, ctor in MODEL_CLASSES.items():
    m = ctor()
    outs = m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
    print(name, "output shapes:", [tuple(o.shape) for o in outs])
    assert outs[0].shape == (2, 1, IMG_SIZE, IMG_SIZE)
    del m, outs


## 7. Loss Function & Metrics

In [ ]:
# ============================================================
# 7a. Tversky (generalizes Dice) + weighted BCE, combined with deep-supervision weighting
# ============================================================
def tversky_loss(logits, targets, alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    TP = (probs * targets).sum(dim=1)
    FP = (probs * (1 - targets)).sum(dim=1)
    FN = ((1 - probs) * targets).sum(dim=1)
    tversky = (TP + eps) / (TP + alpha * FP + beta * FN + eps)
    return 1 - tversky.mean()


class DeepSupervisionLoss(nn.Module):
    def __init__(self, pos_weight=POS_WEIGHT, tversky_wt=0.5, bce_wt=0.5, scale_weights=DS_SCALE_WEIGHTS):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
        self.tversky_wt, self.bce_wt = tversky_wt, bce_wt
        w = torch.tensor(scale_weights, dtype=torch.float32)
        self.register_buffer("scale_weights", w / w.sum())

    def forward(self, outputs, target):
        total = 0.0
        for out, w in zip(outputs, self.scale_weights):
            t = target if out.shape[-2:] == target.shape[-2:] else F.interpolate(target, size=out.shape[-2:], mode="area")
            loss = self.bce_wt * self.bce(out, t) + self.tversky_wt * tversky_loss(out, t)
            total = total + w * loss
        return total

# ============================================================
# 7b. Pixel-level metrics (computed on the MAIN output only)
# ============================================================
@torch.no_grad()
def pixel_dice_iou(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds_f, targets_f = preds.view(preds.size(0), -1), targets.view(targets.size(0), -1)
    intersection = (preds_f * targets_f).sum(dim=1)
    union = preds_f.sum(dim=1) + targets_f.sum(dim=1) - intersection
    dice = (2 * intersection + eps) / (preds_f.sum(dim=1) + targets_f.sum(dim=1) + eps)
    iou = (intersection + eps) / (union + eps)
    return dice.mean().item(), iou.mean().item()

# ============================================================
# 7c. Lesion-level metrics via connected components
# ============================================================
def lesion_level_match(pred_mask, gt_mask, iou_thres=0.10):
    pred_cc = cc_label(pred_mask)
    gt_cc = cc_label(gt_mask)
    pred_regions = regionprops(pred_cc)
    gt_regions = regionprops(gt_cc)
    matched_gt = set()
    TP, FP = 0, 0
    for pr in pred_regions:
        pr_mask = (pred_cc == pr.label)
        best_iou, best_idx = 0.0, -1
        for gi, gr in enumerate(gt_regions):
            if gi in matched_gt:
                continue
            gr_mask = (gt_cc == gr.label)
            inter = np.logical_and(pr_mask, gr_mask).sum()
            union = np.logical_or(pr_mask, gr_mask).sum()
            iou = inter / union if union > 0 else 0.0
            if iou > best_iou:
                best_iou, best_idx = iou, gi
        if best_iou >= iou_thres:
            TP += 1
            matched_gt.add(best_idx)
        else:
            FP += 1
    FN = len(gt_regions) - len(matched_gt)
    return TP, FP, FN

# ============================================================
# 7d. Flip-TTA + post-processing helpers (new, unit-tested)
# ============================================================
def flip_tta_probs(model, img_tensor):
    """img_tensor: (1,3,H,W) on DEVICE. Returns averaged probability map (H,W) numpy."""
    with torch.no_grad():
        probs_orig = torch.sigmoid(model(img_tensor)[0])[0, 0]
        img_flipped = torch.flip(img_tensor, dims=[3])
        probs_flipped = torch.sigmoid(model(img_flipped)[0])[0, 0]
        probs_flipped_back = torch.flip(probs_flipped, dims=[1])
        avg = (probs_orig + probs_flipped_back) / 2.0
    return avg.cpu().numpy()

def filter_small_components(pred_mask, min_area=MIN_COMPONENT_AREA):
    cc = cc_label(pred_mask)
    out = np.zeros_like(pred_mask)
    for r in regionprops(cc):
        if r.area >= min_area:
            out[cc == r.label] = 1
    return out


## 8. Training Loop

Same recipe as v1 (AdamW, cosine LR, mixed precision, early stopping on val Dice, computed from the main output),
now training against `DeepSupervisionLoss` instead of the single-scale loss.


In [ ]:
# ============================================================
# 8. Generic training function
# ============================================================
def train_model(model_name):
    print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")
    model = MODEL_CLASSES[model_name]().to(DEVICE)
    criterion = DeepSupervisionLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    history = []
    best_val_dice = -1.0
    epochs_no_improve = 0
    ckpt_dir = WORK_ROOT / "checkpoints" / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_dir / "best.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss_sum += loss.item() * imgs.size(0)
        train_loss = train_loss_sum / len(train_ds)
        scheduler.step()

        model.eval()
        val_dice_sum, val_iou_sum, n_val = 0.0, 0.0, 0
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                main_out = model(imgs)[0]
                d, i = pixel_dice_iou(main_out, masks)
                val_dice_sum += d * imgs.size(0); val_iou_sum += i * imgs.size(0); n_val += imgs.size(0)
        val_dice, val_iou = val_dice_sum / n_val, val_iou_sum / n_val

        history.append({"epoch": epoch, "train_loss": train_loss, "val_dice": val_dice, "val_iou": val_iou})
        print(f"[{model_name}] epoch {epoch:3d}/{EPOCHS} | train_loss {train_loss:.4f} | "
              f"val_dice {val_dice:.4f} | val_iou {val_iou:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}. Best val_dice = {best_val_dice:.4f}")
                break

    return pd.DataFrame(history), best_path, best_val_dice


In [ ]:
# ============================================================
# 8b. Run training for both models
# ============================================================
histories, best_paths = {}, {}
for model_name in MODELS_TO_TRAIN:
    hist_df, best_path, best_val_dice = train_model(model_name)
    histories[model_name] = hist_df
    best_paths[model_name] = best_path
    print(f"[{model_name}] best val Dice: {best_val_dice:.4f}, checkpoint: {best_path}")


## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for model_name, hist_df in histories.items():
    axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label=model_name)
    axes[1].plot(hist_df["epoch"], hist_df["val_dice"], label=model_name)
    axes[2].plot(hist_df["epoch"], hist_df["val_iou"], label=model_name)
axes[0].set_title("Train Loss"); axes[1].set_title("Val Dice"); axes[2].set_title("Val IoU")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout()
plt.show()


## 10. Evaluate on the Held-out Test Set

Three variants are reported per model checkpoint, so the effect of each addition is visible rather than hidden
inside one final number: **base** (main output, threshold 0.5, no TTA/filtering -- directly comparable to v1),
**+flip-TTA**, and **+flip-TTA+component-filter** (the full v2 pipeline). A fourth row reports the two-model ensemble.


In [ ]:
# ============================================================
# 10a. Load both trained models
# ============================================================
loaded_models = {}
for model_name in MODELS_TO_TRAIN:
    m = MODEL_CLASSES[model_name]().to(DEVICE)
    m.load_state_dict(torch.load(best_paths[model_name], map_location=DEVICE))
    m.eval()
    loaded_models[model_name] = m

# ============================================================
# 10b. Evaluate every variant on the test set
# ============================================================
def evaluate_variant(get_probs_fn, desc):
    """get_probs_fn(fname) -> (H,W) numpy probability map for one test image."""
    dice_sum, iou_sum, n = 0.0, 0.0, 0
    TP, FP, FN = 0, 0, 0
    for fname in tqdm(test_ds.files, desc=desc):
        gt_mask = (cv2.imread(str(MASK_ROOT / "test" / "masks" / fname), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        probs = get_probs_fn(fname)
        pred_mask = (probs > 0.5).astype(np.uint8)

        gt_t = torch.from_numpy(gt_mask.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        pred_t = torch.from_numpy(pred_mask.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        inter = (pred_t * gt_t).sum().item()
        union = pred_t.sum().item() + gt_t.sum().item() - inter
        dice_sum += (2 * inter + 1e-6) / (pred_t.sum().item() + gt_t.sum().item() + 1e-6)
        iou_sum += (inter + 1e-6) / (union + 1e-6)
        n += 1

        tp, fp, fn = lesion_level_match(pred_mask, gt_mask)
        TP += tp; FP += fp; FN += fn

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return {"Pixel_Dice": dice_sum / n, "Pixel_IoU": iou_sum / n,
            "Precision_PPV": precision, "Recall_Sensitivity": recall, "F1": f1}


def load_test_image_tensor(fname):
    img = cv2.cvtColor(cv2.imread(str(MASK_ROOT / "test" / "images" / fname)), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)


results = {}
for model_name, model in loaded_models.items():
    results[f"{model_name}_base"] = evaluate_variant(
        lambda fname, m=model: torch.sigmoid(m(load_test_image_tensor(fname))[0])[0, 0].detach().cpu().numpy(),
        f"{model_name} base")

    results[f"{model_name}_tta"] = evaluate_variant(
        lambda fname, m=model: flip_tta_probs(m, load_test_image_tensor(fname)),
        f"{model_name} +flip-TTA")

    results[f"{model_name}_tta_filtered"] = evaluate_variant(
        lambda fname, m=model: filter_small_components((flip_tta_probs(m, load_test_image_tensor(fname)) > 0.5).astype(np.uint8)).astype(np.float32),
        f"{model_name} +flip-TTA+filter")

# ensemble of both models (average probability maps, with flip-TTA on each)
def ensemble_probs(fname):
    img_t = load_test_image_tensor(fname)
    probs_list = [flip_tta_probs(m, img_t) for m in loaded_models.values()]
    return np.mean(probs_list, axis=0)

results["ensemble_tta_filtered"] = evaluate_variant(
    lambda fname: filter_small_components((ensemble_probs(fname) > 0.5).astype(np.uint8)).astype(np.float32),
    "ensemble (unet+attn) +flip-TTA+filter")

for k, r in results.items():
    print(f"{k:32s} Dice={r['Pixel_Dice']:.4f} IoU={r['Pixel_IoU']:.4f} "
          f"P={r['Precision_PPV']:.4f} R={r['Recall_Sensitivity']:.4f} F1={r['F1']:.4f}")


In [ ]:
# ============================================================
# 10c. Final comparison table -- all v2 variants vs v1 vs YOLO vs literature
# ============================================================
rows = []
for name, r in results.items():
    rows.append({"Method": f"{name} (v2, this notebook)",
                 "Pixel_Dice": round(r["Pixel_Dice"], 4), "Pixel_IoU": round(r["Pixel_IoU"], 4),
                 "Precision_PPV": round(r["Precision_PPV"], 4), "Recall_Sensitivity": round(r["Recall_Sensitivity"], 4),
                 "F1": round(r["F1"], 4)})

rows.append({"Method": "unet (v1, no deep sup./TTA)", "Pixel_Dice": 0.5287, "Pixel_IoU": 0.3832,
             "Precision_PPV": 0.2420, "Recall_Sensitivity": 0.8763, "F1": 0.3793})
rows.append({"Method": "attention_unet (v1, no deep sup./TTA)", "Pixel_Dice": 0.5362, "Pixel_IoU": 0.3934,
             "Precision_PPV": 0.2585, "Recall_Sensitivity": 0.8421, "F1": 0.3956})
rows.append({"Method": "YOLOv8 raw (v4 notebook)", "Pixel_Dice": "-", "Pixel_IoU": "-",
             "Precision_PPV": 0.274, "Recall_Sensitivity": 0.412, "F1": 0.329})
rows.append({"Method": "Official ARCADE YOLOv8 baseline (published)", "Pixel_Dice": "-", "Pixel_IoU": "-",
             "Precision_PPV": "-", "Recall_Sensitivity": "-", "F1": 0.38})
rows.append({"Method": "StenUNet (ARCADE challenge, 3rd place)", "Pixel_Dice": "-", "Pixel_IoU": "-",
             "Precision_PPV": "-", "Recall_Sensitivity": "-", "F1": 0.535})

comparison_df = pd.DataFrame(rows)
pd.set_option("display.width", 160)
print(comparison_df.to_string(index=False))

comparison_path = WORK_ROOT / "v2_comparison_table.csv"
comparison_df.to_csv(comparison_path, index=False)
print("\nSaved to:", comparison_path, "-- open the CSV directly if any column looks cut off when printed.")


## 11. Visualize Predictions vs Ground Truth (Best Variant)

In [ ]:
# ============================================================
# 11. Sample overlays for the best-F1 variant found in Section 10
# ============================================================
best_variant_name = max(results, key=lambda k: results[k]["F1"])
print("Best variant by F1:", best_variant_name, "-> F1 =", round(results[best_variant_name]["F1"], 4))

sample_files = random.sample(test_ds.files, min(6, len(test_ds.files)))
fig, axes = plt.subplots(len(sample_files), 2, figsize=(10, 5 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes[None, :]

for i, fname in enumerate(sample_files):
    img_rgb = cv2.cvtColor(cv2.imread(str(MASK_ROOT / "test" / "images" / fname)), cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(str(MASK_ROOT / "test" / "masks" / fname), cv2.IMREAD_GRAYSCALE)

    gt_overlay = img_rgb.copy(); gt_overlay[gt_mask > 127] = [255, 0, 0]
    axes[i, 0].imshow(cv2.addWeighted(img_rgb, 0.6, gt_overlay, 0.4, 0))
    axes[i, 0].set_title(f"Ground Truth: {fname}"); axes[i, 0].axis("off")

    if best_variant_name.startswith("ensemble"):
        probs = ensemble_probs(fname)
    else:
        model_key = "unet" if best_variant_name.startswith("unet") else "attention_unet"
        probs = flip_tta_probs(loaded_models[model_key], load_test_image_tensor(fname))
    pred_mask = filter_small_components((probs > 0.5).astype(np.uint8)) if "filtered" in best_variant_name else (probs > 0.5).astype(np.uint8)

    pred_overlay = img_rgb.copy(); pred_overlay[pred_mask > 0] = [0, 255, 0]
    axes[i, 1].imshow(cv2.addWeighted(img_rgb, 0.6, pred_overlay, 0.4, 0))
    axes[i, 1].set_title(f"Prediction ({best_variant_name})"); axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


## 12. Save Everything to Google Drive

In [ ]:
for model_name in MODELS_TO_TRAIN:
    out_dir = DRIVE_OUTPUT_DIR / f"seg_v2_{model_name}"
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_paths[model_name], out_dir / "best.pt")
    histories[model_name].to_csv(out_dir / "training_history.csv", index=False)

shutil.copy(comparison_path, DRIVE_OUTPUT_DIR / comparison_path.name)
print("Saved to:", DRIVE_OUTPUT_DIR)


## Notes & Next Steps

- **Report the full Section 10c table**, not just the best row -- the *_base rows are the fair comparison to v1
  (same architecture change, deep supervision + Tversky loss only), and the *_tta / *_tta_filtered / ensemble rows
  show what each additional inference-time trick contributes on top. If a trick doesn't help, that's worth stating
  plainly rather than dropping the row.
- **If precision still hasn't moved much**: check whether `MIN_COMPONENT_AREA` (printed in Section 4b) is actually
  filtering many predicted blobs -- if precision at `_tta_filtered` is barely different from `_tta`, most false
  positives are larger than the filter threshold, meaning they look like plausible lesions in shape/size, not just
  noise speckle, which is a more fundamentally hard error mode than post-processing can fix.
- **If the gap to StenUNet (0.535) is still substantial**: the remaining honest options are (1) `POS_WEIGHT`/
  `TVERSKY_ALPHA` further tuning around the values here, informed by which way *this* run's precision/recall moved,
  not a blind re-guess; (2) multi-preprocessing-variant ensembling at inference (train on raw images too, average
  raw+CLAHE+whatever variant probability maps) -- what StenUNet itself actually does, and a natural next step given
  the U-Net infrastructure here already supports averaging multiple models' probability maps (Section 10b).
- **Objective 1 is complete regardless of where these numbers land** -- report this table as the final Objective-1
  comparison and move to Objective 2 (Grad-CAM) once satisfied with it.
